In [1]:
#pip install mlflow
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

iris = load_iris()
x,y = iris.data,iris.target
x_tr,x_te,y_tr,y_te=train_test_split(x,y,test_size=0.2,random_state=42)
models = {"Logistic Regression" : LogisticRegression(), "Decision Tree": DecisionTreeClassifier(), "Random Forest" : RandomForestClassifier()}
mlflow.set_experiment("Iris Experiment")

reg = "Iris_Classifier"
for name, model in models.items():
    with mlflow.start_run(nested=True):
        model.fit(x_tr, y_tr)
        pred = model.predict(x_te)
        acc = accuracy_score(y_te, pred)
        mlflow.log_param("Model", name)
        mlflow.log_metric("Accuracy", acc)
        mlflow.sklearn.log_model(sk_model = model, name = "model", registered_model_name=reg , input_example= x_tr[:5])
        print("Model:",name)
        print("Accuracy:",acc)
print("Experiment tracking  and model versioning completed")

2026/10/14 21:49:47 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/14 21:49:47 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Successfully registered model 'Iris_Classifier'.
Created version '1' of model 'Iris_Classifier'.


Model: Logistic Regression
Accuracy: 1.0


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo

In [8]:

# MLOps Lab: MLflow Experiment Tracking and Model Versioning

# 1. Import Libraries
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 2. Load Dataset
iris = load_iris()
X = iris.data
y = iris.target

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 4. MLflow Experiment Setup
mlflow.set_experiment("Iris_Experiment_Tracking")

# 5. Define Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# 6. Registered Model Name
registered_model_name = "Iris_Classifier"

# 7. Train and Track Models
for name, model in models.items():
    with mlflow.start_run(run_name=name):

        # Train Model
        model.fit(X_train, y_train)

        # Prediction
        predictions = model.predict(X_test)

        # Calculate Accuracy
        accuracy = accuracy_score(y_test, predictions)

        # Log Parameters
        mlflow.log_param("model_name", name)

        # Log Metrics
        mlflow.log_metric("accuracy", accuracy)

        # Log and Register Model
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=registered_model_name,
            input_example=X_train[:5]
        )

        # Display Results
        print("Model:", name)
        print("Accuracy:", round(accuracy, 4))
        print("-" * 40)

# 8. Completion Message
print("Experiment Tracking and Model Versioning Completed.")


2026/10/14 21:55:54 INFO mlflow.tracking.fluent: Experiment with name 'Iris_Experiment_Tracking' does not exist. Creating a new experiment.
2026/10/14 21:55:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'Iris_Classifier' already exists. Creating a new version of this model...
Created version '2' of model 'Iris_Classifier'.
2026/10/14 21:56:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model: Logistic Regression
Accuracy: 1.0
----------------------------------------


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo